In [ ]:
# ============================================================
# 1000 SAMPLE QUERY TESTER
# Tests 1000 random queries from weather_queries_15M.parquet
# No f-string format specs — safe to copy-paste into JupyterHub
# ============================================================


# ============================================================
# CELL 1 — Load 1000 random samples
# ============================================================

import sys, os
import polars as pl
import pandas as pd

PROJECT_DIR = os.path.join(os.path.expanduser("~"), "imd_api_wrapper")
PARQUET     = os.path.join(os.path.expanduser("~"), "weather_queries_15M.parquet")

sys.path.insert(0, PROJECT_DIR)

from wrapper.router import route_query
from wrapper.client import IMDClient

client = IMDClient()

print("Loading weather_queries_15M.parquet...")
weather_df = pl.read_parquet(PARQUET)
print("Total rows : " + str(len(weather_df)))
print()

sample = (
    weather_df
    .sample(n=1000, seed=42)
    .select(["QueryText", "StateName", "DistrictName", "Season", "Crop"])
    .to_pandas()
)

print("Sample size : " + str(len(sample)))
print("\nFirst 5 queries:")
for i, row in sample.head(5).iterrows():
    print("  " + str(row["QueryText"])[:75])


# ============================================================
# CELL 2 — Route all 1000 queries
# ============================================================

print("Routing 1000 queries...\n")

routing_results = []
for _, row in sample.iterrows():
    query = str(row["QueryText"]) if row["QueryText"] else ""
    route = route_query(query)
    routing_results.append({
        "query"           : query[:80],
        "state"           : str(row.get("StateName",   "") or ""),
        "district"        : str(row.get("DistrictName","") or ""),
        "season"          : str(row.get("Season",      "") or ""),
        "crop"            : str(row.get("Crop",        "") or ""),
        "farmer_need"     : route["farmer_need"],
        "endpoint_key"    : route["endpoint_key"],
        "priority"        : route["priority"],
        "matched_keyword" : str(route["matched_keyword"]),
        "freshness_mins"  : route["freshness_minutes"],
    })

routing_df = pd.DataFrame(routing_results)
print("Done — " + str(len(routing_df)) + " queries routed\n")

print("=== Routing Distribution ===\n")
for need, count in routing_df["farmer_need"].value_counts().items():
    pct = count / len(routing_df) * 100
    bar = "█" * int(pct / 2)
    print("  " + str(need).ljust(35) + str(count).rjust(5) + "  (" + str(round(pct, 1)) + "%)  " + bar)


# ============================================================
# CELL 3 — Call the correct IMD endpoint for each query
# ============================================================

print("\nCalling IMD endpoints for all 1000 queries...\n")

api_results = []
for _, row in routing_df.iterrows():

    ep       = row["endpoint_key"]
    city     = row["state"]    or "Delhi"
    district = row["district"] or "Delhi"
    state    = row["state"]    or "Delhi"
    crop     = row["crop"]     or ""

    try:
        if ep == "city_forecast":
            data = client.get_city_forecast(city=city, state=state)
        elif ep == "district_forecast":
            data = client.get_district_forecast(district=district, state=state)
        elif ep == "rainfall_forecast":
            data = client.get_rainfall_forecast(district=district, state=state, days=5)
        elif ep == "current_weather":
            data = client.get_current_weather(city=city, state=state)
        elif ep == "nowcast":
            data = client.get_nowcast(district=district, state=state)
        elif ep == "agromet_advisory":
            data = client.get_agromet_advisory(district=district, state=state, crop=crop)
        else:
            data = {"error": "Unknown endpoint: " + ep}

        api_results.append({
            "query"        : row["query"],
            "farmer_need"  : row["farmer_need"],
            "endpoint"     : ep,
            "priority"     : row["priority"],
            "status"       : "OK",
            "temperature_c": data.get("temperature_c"),
            "rainfall_mm"  : data.get("rainfall_mm"),
            "humidity_pct" : data.get("humidity_pct"),
            "wind_kmh"     : data.get("wind_speed_kmh"),
            "alert"        : data.get("alert_type", ""),
            "condition"    : data.get("condition", ""),
        })

    except Exception as e:
        api_results.append({
            "query"      : row["query"],
            "farmer_need": row["farmer_need"],
            "endpoint"   : ep,
            "priority"   : row["priority"],
            "status"     : "ERROR",
            "error"      : str(e),
        })

results_df = pd.DataFrame(api_results)
ok  = (results_df["status"] == "OK").sum()
err = (results_df["status"] == "ERROR").sum()
print("Results   : " + str(ok) + " OK  |  " + str(err) + " errors")
print("Success % : " + str(round(ok / len(results_df) * 100, 1)) + "%")


# ============================================================
# CELL 4 — Summary statistics
# ============================================================

ok_df = results_df[results_df["status"] == "OK"]

print("\n=== Summary (1000 queries) ===\n")
print("  Total     : " + str(len(results_df)))
print("  OK        : " + str(ok))
print("  Errors    : " + str(err))
print("  Success % : " + str(round(ok / len(results_df) * 100, 1)) + "%")

temp_vals = ok_df["temperature_c"].dropna()
rain_vals = ok_df["rainfall_mm"].dropna()

if len(temp_vals):
    print("\n  Avg temp  : " + str(round(temp_vals.mean(), 1)) + " C")
    print("  Min temp  : " + str(round(temp_vals.min(), 1)) + " C")
    print("  Max temp  : " + str(round(temp_vals.max(), 1)) + " C")

if len(rain_vals):
    print("\n  Avg rain  : " + str(round(rain_vals.mean(), 1)) + " mm")
    print("  Max rain  : " + str(round(rain_vals.max(), 1)) + " mm")

print("\n=== Endpoint Usage ===\n")
for ep, count in results_df["endpoint"].value_counts().items():
    print("  " + str(ep).ljust(28) + str(count).rjust(5) + "  (" + str(round(count / len(results_df) * 100, 1)) + "%)")

print("\n=== Priority Breakdown ===\n")
for p, count in results_df["priority"].value_counts().items():
    print("  " + str(p).ljust(12) + str(count).rjust(5) + "  (" + str(round(count / len(results_df) * 100, 1)) + "%)")


# ============================================================
# CELL 5 — Show 20 random results (fixed — no f-string specs)
# ============================================================

print("\n=== 20 Random Results ===\n")
print("  St    Query                                          Routed To                     Temp    Rain")
print("  " + "-" * 100)

sample_20 = results_df.sample(20, random_state=7)

for i, (_, row) in enumerate(sample_20.iterrows(), 1):
    mark  = "OK " if row["status"] == "OK" else "ERR"
    query = str(row["query"])[:44].ljust(45)
    need  = str(row["farmer_need"])[:27].ljust(28)

    temp_val = row.get("temperature_c")
    rain_val = row.get("rainfall_mm")
    temp = (str(round(temp_val, 1)) + "C") if pd.notna(temp_val) else "—"
    rain = str(round(rain_val, 1))          if pd.notna(rain_val) else "—"

    line = "  [" + mark + "] " + query + "  " + need + "  " + temp.rjust(6) + "  " + rain.rjust(6)
    print(line)


# ============================================================
# CELL 6 — Show errors if any
# ============================================================

err_df = results_df[results_df["status"] == "ERROR"]
if len(err_df) == 0:
    print("\nNo errors — all 1000 queries processed successfully ✓")
else:
    print("\n" + str(len(err_df)) + " errors:\n")
    for _, row in err_df.head(10).iterrows():
        print("  Query    : " + str(row["query"])[:60])
        print("  Endpoint : " + str(row["endpoint"]))
        print("  Error    : " + str(row.get("error", "")))
        print()


# ============================================================
# CELL 7 — Save results to CSV
# ============================================================

out1 = os.path.join(PROJECT_DIR, "sample_1000_test_results.csv")
out2 = os.path.join(PROJECT_DIR, "sample_1000_routing.csv")

results_df.to_csv(out1, index=False)
routing_df.to_csv(out2, index=False)

print("Saved → sample_1000_test_results.csv  (" + str(len(results_df)) + " rows)")
print("Saved → sample_1000_routing.csv       (" + str(len(routing_df)) + " rows)")
print("\nColumns in results file:")
print("  " + str(list(results_df.columns)))